# PhasePhyto Apple Overlap Fixes v2 (Oracle calibration + softer rebalancing)

Iteration 2 of the apple-overlap fixes pipeline. The v1 notebook
(`PhasePhyto_Apple_Overlap_Fixes_Colab.ipynb`) found:

- **Fix A with uniform target prior was net-negative** because PP2021 isn't
  uniform.
- **Fix B with `balanced_sampler_power=1.0` lifted scab and healthy on
  PP2021 but hurt rust** because the small PV rust corpus (217 images) was
  had its sampled share lifted from 10.6% to 33.3% per epoch (3.13x oversampling), enough to memorize.

This v2 notebook applies two corrections that target those exact failure
modes:

- **Fix A oracle:** logit shift `log(p_target) - log(p_source)` where
  `p_target` is estimated from PP2021 labels. Upper bound on prior
  correction. Uses target labels, so this is an analysis tool, not a
  deployable configuration.
- **Fix B softer:** new committed config
  `configs/apple_overlap_plantdoc_rebalanced_softer.yaml` with
  `balanced_sampler: true` and `balanced_sampler_power: 0.5` (sqrt-softened
  inverse frequency). Lifts rust exposure ~1.88x instead of ~3.13x.

> **If you edit any cell, click `Runtime -> Restart runtime` and `Run all`.**

Prerequisites (produced by earlier notebooks):

- `MyDrive/PhasePhyto/data/archives/apple_strict.tar` (from
  `PhasePhyto_Apple_Overlap_Colab.ipynb`)
- `MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc/best_model.pt`
  (the PV-trained baseline checkpoint, also from the apple-overlap notebook)
- The v1 fix evals are not required for v2 to run, but if present the
  comparison cell will stack v1 and v2 results side by side.


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    "repo_url": "https://github.com/kautilyaa/PhasePhyto.git",
    "repo_branch": "new_updation",
    "repo_dir": "/content/PhasePhyto",
    # v1 baseline (read-only here -- produced by the apple-overlap notebook).
    "baseline_checkpoint_dir": "/content/drive/MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc",
    # v1 Fix B retrain (read for the comparison cell only; not produced here).
    "fix_b_full_checkpoint_dir": "/content/drive/MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc_rebalanced",
    # v2 Fix B softer (produced by this notebook).
    "fix_b_softer_checkpoint_dir": "/content/drive/MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc_rebalanced_softer",
    # v2 comparison artifacts go in their own folder so v1 stays intact.
    "comparison_dir": "/content/drive/MyDrive/PhasePhyto/checkpoints/apple_overlap_fixes_v2_comparison",
    "run_fix_a_oracle": True,
    "run_fix_b_softer_retrain": True,
    "run_fix_b_softer_eval": True,
    "run_comparison": True,
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")


In [ ]:
# ============================================================
# MOUNT DRIVE + CLONE/INSTALL REPO + DEFINE PATHS
# ============================================================

from pathlib import Path
import os
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path(CONFIG["repo_dir"])
if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", CONFIG["repo_branch"],
        CONFIG["repo_url"], str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", CONFIG["repo_branch"], "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DRIVE_PROJECT_DIR = Path(CONFIG["drive_project_dir"])
ARCHIVE_DIR = DRIVE_PROJECT_DIR / "data" / "archives"
OVERLAP_ARCHIVE = ARCHIVE_DIR / "apple_strict.tar"
LOCAL_DATA_ROOT = Path("/content/data")
LOCAL_OVERLAP_PARENT = LOCAL_DATA_ROOT / "overlap"
LOCAL_OVERLAP_ROOT = LOCAL_OVERLAP_PARENT / "apple_strict"

BASELINE_CKPT_DIR = Path(CONFIG["baseline_checkpoint_dir"])
FIX_B_FULL_CKPT_DIR = Path(CONFIG["fix_b_full_checkpoint_dir"])
FIX_B_SOFTER_CKPT_DIR = Path(CONFIG["fix_b_softer_checkpoint_dir"])
COMPARISON_DIR = Path(CONFIG["comparison_dir"])

for path in [DRIVE_PROJECT_DIR, ARCHIVE_DIR, LOCAL_DATA_ROOT, COMPARISON_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Repo ready:", REPO_DIR)
print("Overlap archive:", OVERLAP_ARCHIVE)
print("Baseline ckpt dir:", BASELINE_CKPT_DIR)
print("Fix B full ckpt dir:", FIX_B_FULL_CKPT_DIR)
print("Fix B softer ckpt dir:", FIX_B_SOFTER_CKPT_DIR)
print("Comparison dir:", COMPARISON_DIR)


In [ ]:
# ============================================================
# HELPERS
# ============================================================

import json
import shutil
import sys
import tarfile
from pathlib import Path

CONFIGS_DIR = REPO_DIR / "configs"
PLANTDOC_CFG = CONFIGS_DIR / "apple_overlap_plantdoc.yaml"
PP2021_CFG = CONFIGS_DIR / "apple_overlap_pp2021.yaml"
REBALANCED_SOFTER_CFG = CONFIGS_DIR / "apple_overlap_plantdoc_rebalanced_softer.yaml"
for cfg in (PLANTDOC_CFG, PP2021_CFG, REBALANCED_SOFTER_CFG):
    if not cfg.exists():
        raise FileNotFoundError(
            f"Missing committed config: {cfg}. Make sure CONFIG['repo_branch'] "
            f"({CONFIG['repo_branch']}) contains the v2 configs."
        )


def run(cmd, *, capture: bool = False):
    print("\n$ " + " ".join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=False,
        text=True,
        capture_output=capture,
    )
    if capture and result.stdout:
        print(result.stdout)
    if capture and result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): " + " ".join(map(str, cmd))
        )
    return result


def extract_tar_to_parent(archive_path: Path, parent_dir: Path, *, clean_target: bool = True) -> Path:
    if not archive_path.exists():
        raise FileNotFoundError(f"Archive not found: {archive_path}")
    with tarfile.open(archive_path) as tf:
        top_levels = [m.name.split("/", 1)[0] for m in tf.getmembers() if m.name]
        top_level = next((n for n in top_levels if n and n != "."), None)
        if top_level is None:
            raise RuntimeError(f"Could not determine top-level folder inside {archive_path}")
    target_dir = parent_dir / top_level
    if clean_target and target_dir.exists():
        shutil.rmtree(target_dir)
    parent_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path) as tf:
        tf.extractall(parent_dir)
    print(f"Extracted {archive_path} -> {target_dir}")
    return target_dir


## Preflight: hydrate baseline data + check baseline checkpoint


In [ ]:
# ============================================================
# HYDRATE OVERLAP DATA + VERIFY BASELINE
# ============================================================

if not OVERLAP_ARCHIVE.exists():
    raise FileNotFoundError(
        f"Overlap archive missing: {OVERLAP_ARCHIVE}. "
        "Run PhasePhyto_Apple_Overlap_Colab.ipynb first to produce it."
    )

if not LOCAL_OVERLAP_ROOT.exists() or not (LOCAL_OVERLAP_ROOT / "plantvillage").exists():
    if LOCAL_OVERLAP_PARENT.exists():
        shutil.rmtree(LOCAL_OVERLAP_PARENT)
    extract_tar_to_parent(OVERLAP_ARCHIVE, LOCAL_OVERLAP_PARENT, clean_target=False)
    print("Hydrated overlap archive to SSD:", LOCAL_OVERLAP_ROOT)
else:
    print("Overlap data already on SSD:", LOCAL_OVERLAP_ROOT)

baseline_ckpt = BASELINE_CKPT_DIR / "best_model.pt"
if not baseline_ckpt.exists():
    raise FileNotFoundError(
        f"Baseline checkpoint not found: {baseline_ckpt}. "
        "Run PhasePhyto_Apple_Overlap_Colab.ipynb with run_train=True first."
    )
print(f"Baseline checkpoint: {baseline_ckpt} ({baseline_ckpt.stat().st_size / 1e6:.1f} MB)")


## Fix A oracle: logit adjustment with the actual PP2021 prior

Same recipe as v1 Fix A, but `p_target` is now estimated from PP2021 labels
instead of assumed uniform. This uses target labels and is therefore an
upper bound, not a deployable configuration.


In [ ]:
# ============================================================
# FIX A ORACLE: BUILD MODEL FROM BASELINE CHECKPOINT, COLLECT LOGITS,
# APPLY PP2021-PRIOR-AWARE LOGIT ADJUSTMENT
# ============================================================

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader

from phasephyto.data.datasets import PlantDiseaseDataset
from phasephyto.data.transforms import get_val_transforms
from phasephyto.models import PhasePhyto
from phasephyto.utils.config import load_config


def collect_logits(model, loader, device):
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                rgb, clahe, targets = batch
                rgb = rgb.to(device, non_blocking=True)
                clahe = clahe.to(device, non_blocking=True)
            else:
                rgb, targets = batch
                rgb = rgb.to(device, non_blocking=True)
                clahe = None
            outputs = model(rgb, x_clahe=clahe)
            if isinstance(outputs, dict):
                logits = outputs["logits"]
            elif isinstance(outputs, (tuple, list)):
                logits = outputs[0]
            else:
                logits = outputs
            all_logits.append(logits.detach().cpu())
            all_targets.append(targets)
    return torch.cat(all_logits).numpy(), torch.cat(all_targets).numpy()


def metrics_from_preds(targets, preds, class_names):
    return {
        "accuracy": float(accuracy_score(targets, preds)),
        "f1_macro": float(f1_score(targets, preds, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(targets, preds, average="weighted", zero_division=0)),
        "precision_macro": float(precision_score(targets, preds, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(targets, preds, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(targets, preds, labels=list(range(len(class_names)))).tolist(),
        "report": classification_report(
            targets, preds,
            labels=list(range(len(class_names))),
            target_names=class_names,
            zero_division=0,
        ),
    }


if CONFIG["run_fix_a_oracle"]:
    cfg = load_config(str(PLANTDOC_CFG))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    val_tf = get_val_transforms(cfg.data.image_size)

    pv_root = LOCAL_OVERLAP_ROOT / "plantvillage"
    pp_root = LOCAL_OVERLAP_ROOT / "plant_pathology_2021"

    source_ds = PlantDiseaseDataset(root=pv_root, transform=val_tf)
    target_ds = PlantDiseaseDataset(root=pp_root, transform=val_tf, class_to_idx=source_ds.class_to_idx)
    class_names = source_ds.classes
    num_classes = source_ds.num_classes

    print(f"Classes: {class_names}")
    print(f"PV samples: {len(source_ds)} | PP2021 samples: {len(target_ds)}")

    pv_loader = DataLoader(source_ds, batch_size=cfg.training.batch_size,
                           num_workers=2, pin_memory=True)
    pp_loader = DataLoader(target_ds, batch_size=cfg.training.batch_size,
                           num_workers=2, pin_memory=True)

    model = PhasePhyto(
        num_classes=num_classes,
        backbone_name=cfg.model.backbone_name,
        fusion_dim=cfg.model.fusion_dim,
        pc_scales=cfg.model.pc_scales,
        pc_orientations=cfg.model.pc_orientations,
        image_size=(cfg.data.image_size, cfg.data.image_size),
        num_heads=cfg.model.num_heads,
        dropout=cfg.model.dropout,
    ).to(device)

    state = torch.load(baseline_ckpt, map_location=device, weights_only=True)
    model.load_state_dict(state["model_state_dict"])
    print(f"Loaded baseline checkpoint (epoch {state.get('epoch', '?')}).")

    print("Collecting PV logits...")
    pv_logits, pv_targets = collect_logits(model, pv_loader, device)
    print("Collecting PP2021 logits...")
    pp_logits, pp_targets = collect_logits(model, pp_loader, device)

    source_counts = np.bincount(pv_targets, minlength=num_classes).astype(float)
    target_counts = np.bincount(pp_targets, minlength=num_classes).astype(float)
    source_prior = source_counts / source_counts.sum()
    target_prior = target_counts / target_counts.sum()
    log_source_prior = np.log(np.clip(source_prior, 1e-12, None))
    log_target_prior = np.log(np.clip(target_prior, 1e-12, None))
    shift = log_target_prior - log_source_prior

    print(f"\nSource prior: {dict(zip(class_names, source_prior.round(3)))}")
    print(f"Target prior (oracle, PP2021 labels): {dict(zip(class_names, target_prior.round(3)))}")
    print(f"Logit shift: {dict(zip(class_names, shift.round(3)))}")

    pp_preds_baseline = pp_logits.argmax(axis=1)
    pp_preds_oracle = (pp_logits + shift).argmax(axis=1)
    pv_preds_oracle = (pv_logits + shift).argmax(axis=1)

    fix_a_oracle = {
        "source": metrics_from_preds(pv_targets, pv_preds_oracle, class_names),
        "target": metrics_from_preds(pp_targets, pp_preds_oracle, class_names),
        "delta": {},
        "use_case": "plant_disease",
        "calibration": {
            "method": "logit_adjustment",
            "target_prior_source": "oracle (PP2021 label distribution)",
            "source_prior": source_prior.tolist(),
            "target_prior": target_prior.tolist(),
            "logit_shift": shift.tolist(),
        },
    }
    fix_a_oracle["delta"]["accuracy_drop"] = (
        fix_a_oracle["target"]["accuracy"] - fix_a_oracle["source"]["accuracy"]
    )
    fix_a_oracle["delta"]["f1_drop"] = (
        fix_a_oracle["target"]["f1_macro"] - fix_a_oracle["source"]["f1_macro"]
    )

    out_path = BASELINE_CKPT_DIR / "eval_pp2021_calibrated_oracle.json"
    out_path.write_text(json.dumps(fix_a_oracle, indent=2, default=str) + "\n")
    print(f"\nFix A oracle eval written to: {out_path}")

    print("\nBaseline vs Fix A oracle on PP2021:")
    print(f"  acc:  baseline={accuracy_score(pp_targets, pp_preds_baseline):.4f} -> oracle={fix_a_oracle['target']['accuracy']:.4f}")
    print(f"  f1:   baseline={f1_score(pp_targets, pp_preds_baseline, average='macro', zero_division=0):.4f} -> oracle={fix_a_oracle['target']['f1_macro']:.4f}")
    print("\nPer-class report (Fix A oracle on PP2021):")
    print(fix_a_oracle["target"]["report"])
else:
    print("Set CONFIG['run_fix_a_oracle'] = True to run the oracle Fix A.")


## Fix B softer: rebalanced retrain with `balanced_sampler_power=0.5`

Drives `python -m phasephyto.train --config configs/apple_overlap_plantdoc_rebalanced_softer.yaml`.
The committed config flips `balanced_sampler_power` to `0.5` (sqrt-softened).
Default off because it needs a GPU and ~30-45 min on a T4.


In [ ]:
# ============================================================
# FIX B SOFTER: REBALANCED RETRAIN WITH balanced_sampler_power=0.5
# ============================================================

FIX_B_SOFTER_CKPT_DIR.mkdir(parents=True, exist_ok=True)

if CONFIG["run_fix_b_softer_retrain"]:
    run([
        sys.executable, "-m", "phasephyto.train",
        "--config", str(REBALANCED_SOFTER_CFG),
        "--override",
        f"checkpoint_dir={FIX_B_SOFTER_CKPT_DIR}",
        f"data.root={LOCAL_OVERLAP_ROOT}",
        f"data.source_dir={LOCAL_OVERLAP_ROOT / 'plantvillage'}",
        f"data.target_dir={LOCAL_OVERLAP_ROOT / 'plantdoc'}",
        f"data.eval_source_dir={LOCAL_OVERLAP_ROOT / 'plantvillage'}",
        f"data.eval_target_dir={LOCAL_OVERLAP_ROOT / 'plantdoc'}",
    ], capture=True)
else:
    print("Set CONFIG['run_fix_b_softer_retrain'] = True to launch the softer retrain.")


In [ ]:
# ============================================================
# FIX B SOFTER: EVAL ON PD AND PP2021
# ============================================================

fix_b_softer_ckpt = FIX_B_SOFTER_CKPT_DIR / "best_model.pt"

if CONFIG["run_fix_b_softer_eval"]:
    if not fix_b_softer_ckpt.exists():
        raise FileNotFoundError(
            f"Softer Fix B checkpoint missing: {fix_b_softer_ckpt}. "
            "Set CONFIG['run_fix_b_softer_retrain']=True and rerun."
        )
    run([
        sys.executable, "-m", "phasephyto.evaluate",
        "--config", str(REBALANCED_SOFTER_CFG),
        "--checkpoint", str(fix_b_softer_ckpt),
        "--source-dir", str(LOCAL_OVERLAP_ROOT / "plantvillage"),
        "--target-dir", str(LOCAL_OVERLAP_ROOT / "plantdoc"),
        "--output", str(FIX_B_SOFTER_CKPT_DIR / "eval_plantdoc.json"),
    ], capture=True)
    run([
        sys.executable, "-m", "phasephyto.evaluate",
        "--config", str(PP2021_CFG),
        "--checkpoint", str(fix_b_softer_ckpt),
        "--source-dir", str(LOCAL_OVERLAP_ROOT / "plantvillage"),
        "--target-dir", str(LOCAL_OVERLAP_ROOT / "plant_pathology_2021"),
        "--output", str(FIX_B_SOFTER_CKPT_DIR / "eval_pp2021.json"),
    ], capture=True)
else:
    print("Set CONFIG['run_fix_b_softer_eval'] = True to evaluate the softer checkpoint.")


## Comparison: baseline vs Fix A (uniform) vs Fix A (oracle) vs Fix B (full) vs Fix B (softer)

Stacks every available variant for both PP2021 and PlantDoc targets.


In [ ]:
# ============================================================
# FIVE-WAY COMPARISON ON PP2021 AND PD
# ============================================================

import numpy as np
import pandas as pd


def load_eval(path):
    if not path.exists():
        return None
    return json.loads(path.read_text())


def per_class_rows(label, eval_payload, class_names):
    if eval_payload is None:
        return []
    cm = np.array(eval_payload["target"]["confusion_matrix"])
    rows = []
    for i, cls in enumerate(class_names):
        support = int(cm[i].sum())
        tp = int(cm[i, i])
        recall = tp / support if support else 0.0
        col_sum = int(cm[:, i].sum())
        precision = tp / col_sum if col_sum else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({
            "variant": label,
            "class": cls,
            "support": support,
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
        })
    return rows


def macro_row(label, eval_payload):
    if eval_payload is None:
        return None
    return {
        "variant": label,
        "target_accuracy": round(eval_payload["target"]["accuracy"], 4),
        "target_f1_macro": round(eval_payload["target"]["f1_macro"], 4),
        "target_precision_macro": round(eval_payload["target"]["precision_macro"], 4),
        "target_recall_macro": round(eval_payload["target"]["recall_macro"], 4),
    }


if CONFIG["run_comparison"]:
    APPLE_CLASSES = ["Apple___Apple_scab", "Apple___Cedar_apple_rust", "Apple___healthy"]

    # PP2021 evals
    pp_paths = {
        "baseline":               BASELINE_CKPT_DIR  / "eval_pp2021.json",
        "fix_a_uniform":          BASELINE_CKPT_DIR  / "eval_pp2021_calibrated.json",
        "fix_a_oracle":           BASELINE_CKPT_DIR  / "eval_pp2021_calibrated_oracle.json",
        "fix_b_full_rebalance":   FIX_B_FULL_CKPT_DIR / "eval_pp2021.json",
        "fix_b_softer_rebalance": FIX_B_SOFTER_CKPT_DIR / "eval_pp2021.json",
    }
    # PlantDoc evals (Fix A variants don't have separate PD evals; they share
    # the baseline behavior on PD because logit shift was only applied to PP2021)
    pd_paths = {
        "baseline":               BASELINE_CKPT_DIR  / "eval_plantdoc.json",
        "fix_b_full_rebalance":   FIX_B_FULL_CKPT_DIR / "eval_plantdoc.json",
        "fix_b_softer_rebalance": FIX_B_SOFTER_CKPT_DIR / "eval_plantdoc.json",
    }

    print("=== PP2021 macro comparison ===")
    pp_rows = [r for r in (macro_row(l, load_eval(p)) for l, p in pp_paths.items()) if r]
    if pp_rows:
        pp_macro_df = pd.DataFrame(pp_rows)
        print(pp_macro_df.to_string(index=False))
    else:
        print("(no PP2021 evals available yet)")

    print("\n=== PlantDoc macro comparison ===")
    pd_rows = [r for r in (macro_row(l, load_eval(p)) for l, p in pd_paths.items()) if r]
    if pd_rows:
        pd_macro_df = pd.DataFrame(pd_rows)
        print(pd_macro_df.to_string(index=False))
    else:
        print("(no PlantDoc evals available yet)")

    print("\n=== PP2021 per-class precision/recall/F1 ===")
    pp_pc = sum((per_class_rows(l, load_eval(p), APPLE_CLASSES) for l, p in pp_paths.items()), [])
    if pp_pc:
        pp_pc_df = pd.DataFrame(pp_pc)
        wide = pp_pc_df.pivot(index="class", columns="variant", values=["precision", "recall", "f1"])
        print(wide.round(4).to_string())

    COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
    if pp_rows:
        pd.DataFrame(pp_rows).to_csv(COMPARISON_DIR / "pp2021_macro_comparison_v2.csv", index=False)
    if pd_rows:
        pd.DataFrame(pd_rows).to_csv(COMPARISON_DIR / "plantdoc_macro_comparison_v2.csv", index=False)
    if pp_pc:
        pd.DataFrame(pp_pc).to_csv(COMPARISON_DIR / "pp2021_per_class_comparison_v2.csv", index=False)
    (COMPARISON_DIR / "comparison_summary_v2.json").write_text(
        json.dumps({"pp2021_macro": pp_rows, "plantdoc_macro": pd_rows}, indent=2) + "\n"
    )
    print(f"\nSaved v2 comparison artifacts under: {COMPARISON_DIR}")
else:
    print("Set CONFIG['run_comparison'] = True to build the comparison table.")


## Reading the comparison

If the v2 fixes work as predicted:

- **Fix A oracle** should beat Fix A uniform on PP2021 (and probably the
  baseline too on macro-F1). If it *doesn't* — meaning even the oracle
  can't recover much — the gap is provably feature-level rather than
  prior-level. Either way is a clean writeup figure.
- **Fix B softer** (`balanced_sampler_power=0.5`) should keep most of the
  scab/healthy gains from Fix B full, but recover some rust performance.
  Specifically: rust F1 in Fix B full was 0.56 (vs baseline 0.60). Softer
  weighting should land between those, ideally above 0.60 if it works.

If neither v2 fix moves the rust number meaningfully, the residual gap is
a true feature-shift problem and the next lever would be transductive
(TENT on PP2021 or a small target fine-tune).

Single seed throughout. PD-target n=29 still anecdotal. PP2021 n=11,310
remains the statistically meaningful target.
